# trend_participation_bc_warmstart_screen_v1

Purpose: take the non-leaking trend participation target contract from `trend_participation_foldlocal_target_v1` and test whether a behavior-cloning warm-start actor can actually learn it without worsening walk-forward return/drawdown versus the baseline fold-local contract.

Governance:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `trend_participation_foldlocal_target_v1` and the unchanged `baseline_no_sequence_cap_target_contract.csv`.
- **Role**: `diagnose` / BC warm-start screen; no runtime promotion and no actor retention.
- **Target windows**: `wf_2022` and `wf_2023_2024`. `wf_2020` is skipped because the target contract starts in 2020, which would leak that test year into BC labels.
- **Changed layer**: `model_selection` / `training_target` / `BC warm-start`; no reward, architecture, or broker change.
- **External reference**: Qlib for repeatable ML evaluation, FinRL for RL/action separation, DAgger/behavior-cloning literature for why supervised warm-start is only a pre-policy-improvement diagnostic, PBO for promotion discipline, and Deep Momentum Networks for sequence models in trend-following settings.
- **Hypothesis**: if the trend target labels are useful rather than overfit, a fold-local BC actor trained on the trend contract should preserve or improve walk-forward relative return while keeping MaxDD <= 15% and drawdown drift versus the baseline BC actor within +2pp.
- **Acceptance criteria**: non-baseline candidate passes `relative_return >= +30pp`, `MaxDD <= 15%`, trend/sparse slice fit, and drift gate on every evaluated fold. Passing this card only allows the next checkpoint/PPO screen; promotion remains blocked until fixed-split and full walk-forward replay.
- **Failure mode**: trend labels fit in-sample but fail held-out fold gates, or improve return while widening MaxDD; then do not continue PPO from this target.
- **Comparison-table row key**: `trend_participation_bc_warmstart_screen_v1`.

Historical reflection:

- Closest success: `cap10_joint_contract_bc_slice_screen_v1` showed fold-local contracts can pass BC slice-fit readiness before PPO.
- Closest failure: `sparse_release_bc_ppo_screen_v1` and `cap10_joint_contract_bc_ppo_drift_screen_v1` showed PPO/checkpoint continuation can widen MaxDD even when BC initialization is useful.
- Why this is not a duplicate: this screen tests the newly added high-trend/low-volatility participation target rows against a baseline BC actor, with fold-local target rows and an explicit drift gate.


In [1]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
os.chdir(ROOT)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.dashboard import write_review_daily_csv
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    score_bc_walkforward_drift,
    summarize_bc_walkforward_drift_screen,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'trend_participation_bc_warmstart_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / 'runtime/model_selection/card/trend_participation_foldlocal_target_v1'
BASE_TARGETS_PATH = ROOT / 'runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv'
TREND_TARGETS_PATH = SOURCE_CARD / 'selected_trend_foldlocal_targets.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DRIFT_RELATIVE_FLOOR = -5.0
DRIFT_MAX_DD_CEILING = 2.0
GLOBAL_ACCURACY_MIN = 0.70
SPARSE_SLICE_ACCURACY_MIN = 0.60
TREND_SLICE_MIN_SAMPLES = 5
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
        'skip_reason': '',
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
        'skip_reason': '',
    },
}

CANDIDATES = [
    {
        'candidate': 'base_plain_contract',
        'target_source': 'base',
        'target_path': BASE_TARGETS_PATH,
        'complexity': 0,
        'sample_weight_config': None,
        'description': 'Unchanged fold-local contract with plain BC; drift baseline.',
    },
    {
        'candidate': 'trend_plain_contract',
        'target_source': 'trend',
        'target_path': TREND_TARGETS_PATH,
        'complexity': 1,
        'sample_weight_config': None,
        'description': 'Trend target contract with plain BC.',
    },
    {
        'candidate': 'trend_weighted_obj025_cb099',
        'target_source': 'trend',
        'target_path': TREND_TARGETS_PATH,
        'complexity': 2,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=1.5,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=20.0,
        ),
        'description': 'Trend target contract with moderate sparse-state and action-tail weighting.',
    },
]
BASELINE_CANDIDATE = 'base_plain_contract'

EXTERNAL_SOURCES = [
    'https://github.com/microsoft/qlib',
    'https://arxiv.org/abs/2009.11189',
    'https://github.com/AI4Finance-Foundation/FinRL',
    'https://arxiv.org/abs/2011.09607',
    'https://proceedings.mlr.press/v15/ross11a.html',
    'https://arxiv.org/abs/1904.04912',
    'https://doi.org/10.21314/JCF.2016.322',
]

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/trend_participation_bc_warmstart_screen_v1


In [2]:
def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=['datetime'])
    bool_cols = [
        'is_rebound_cluster',
        'is_downtrend_state',
        'release_gate_state',
        'drawdown_control_state',
        'sequence_risk_state',
        'trend_participation_signal_ready',
        'trend_participation_guard_ready',
        'trend_participation_target_state',
    ]
    for col in bool_cols:
        if col not in frame.columns:
            frame[col] = False
        frame[col] = frame[col].fillna(False).astype(bool)
    if 'rebound_participation_weight' not in frame.columns:
        frame['rebound_participation_weight'] = 0.0
    return frame


def fold_target_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec['train'][1])
    subset = targets[
        targets['fold'].astype(str).eq(str(spec['target_fold']))
        & (pd.to_datetime(targets['datetime']) <= train_end)
    ].copy()
    return subset.sort_values('datetime').reset_index(drop=True)

source_selection = json.loads(SOURCE_SELECTION_PATH.read_text(encoding='utf-8'))
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
base_targets = load_targets(BASE_TARGETS_PATH)
trend_targets = load_targets(TREND_TARGETS_PATH)
targets_by_source = {'base': base_targets, 'trend': trend_targets}

config_rows = []
coverage_rows = []
for item in CANDIDATES:
    cfg = item['sample_weight_config']
    config_rows.append({
        'candidate': item['candidate'],
        'target_source': item['target_source'],
        'complexity': int(item['complexity']),
        'sample_weight_config': None if cfg is None else asdict(cfg),
        'description': item['description'],
    })
    targets = targets_by_source[item['target_source']]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        subset = fold_target_subset(targets, spec)
        coverage_rows.append({
            'candidate': item['candidate'],
            'fold': fold_name,
            'target_fold': spec['target_fold'],
            'target_rows': int(len(subset)),
            'unique_target_dates': int(subset['datetime'].nunique()) if len(subset) else 0,
            'trend_target_rows': int(subset['trend_participation_target_state'].sum()) if len(subset) else 0,
            'rebound_rows': int(subset['is_rebound_cluster'].sum()) if len(subset) else 0,
            'downtrend_rows': int(subset['is_downtrend_state'].sum()) if len(subset) else 0,
            'train_start': spec['train'][0],
            'train_end': spec['train'][1],
            'valid_start': spec['valid'][0],
            'valid_end': spec['valid'][1],
            'test_start': spec['test'][0],
            'test_end': spec['test'][1],
        })

candidate_configs = pd.DataFrame(config_rows)
coverage = pd.DataFrame(coverage_rows)
candidate_configs.to_csv(CARD / 'bc_candidate_configs.csv', index=False)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_experiment': 'trend_participation_foldlocal_target_v1',
    'source_decision': source_selection.get('decision'),
    'source_training_allowed_from_this_card': bool(source_selection.get('training_allowed_from_this_card', False)),
    'baseline_targets': str(BASE_TARGETS_PATH.relative_to(ROOT)),
    'trend_targets': str(TREND_TARGETS_PATH.relative_to(ROOT)),
    'frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'windows': coverage_rows,
    'skipped_windows': [
        {'fold': 'wf_2020', 'reason': 'target rows start in 2020; using them would leak the wf_2020 test window'}
    ],
})
display(candidate_configs)
display(coverage)


,candidate,target_source,complexity,sample_weight_config,description
0,base_plain_contract,base,0,None,Unchanged fold-local contract with plain BC; d...
1,trend_plain_contract,trend,1,None,Trend target contract with plain BC.
2,trend_weighted_obj025_cb099,trend,2,"{'rebound_multiplier': 4.0, 'downtrend_multipl...",Trend target contract with moderate sparse-sta...


,candidate,fold,target_fold,target_rows,unique_target_dates,trend_target_rows,rebound_rows,downtrend_rows,train_start,train_end,valid_start,valid_end,test_start,test_end
0,base_plain_contract,wf_2022,wf_2022_train,253,253,0,66,118,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,base_plain_contract,wf_2023_2024,wf_2023_2024_train,756,756,0,66,478,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
2,trend_plain_contract,wf_2022,wf_2022_train,253,253,0,66,118,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
3,trend_plain_contract,wf_2023_2024,wf_2023_2024_train,756,756,15,81,478,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
4,trend_weighted_obj025_cb099,wf_2022,wf_2022_train,253,253,0,66,118,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
5,trend_weighted_obj025_cb099,wf_2023_2024,wf_2023_2024_train,756,756,15,81,478,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
def price_slice(dataset, segment: tuple[str, str]) -> pd.Series:
    return dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().loc[segment[0]:segment[1]]


def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str], *, label: str) -> tuple[dict[str, float], Path]:
    actor.eval()
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_sharpe'] = float(bt.get('sharpe', out.get('sharpe', 0.0)))
    daily_path = CARD / f'{label}_daily_path.csv'
    write_review_daily_csv(
        daily_path,
        bt['portfolio'],
        price_series,
        bt.get('position_history', []),
        bt.get('trade_signals', []),
    )
    return {k: float(v) for k, v in out.items() if isinstance(v, (int, float, np.integer, np.floating))}, daily_path


def fit_slice_gate(slice_metrics: pd.DataFrame, *, trend_samples: int) -> tuple[bool, dict[str, Any]]:
    metrics = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    failed = []
    details = {}
    if 'all' in metrics.index:
        all_acc = float(metrics.loc['all', 'accuracy'])
        details['all_accuracy'] = all_acc
        if not math.isfinite(all_acc) or all_acc < GLOBAL_ACCURACY_MIN:
            failed.append('all_accuracy')
    else:
        failed.append('all_slice_missing')
    if trend_samples >= TREND_SLICE_MIN_SAMPLES:
        if 'trend_participation_target_state' not in metrics.index:
            failed.append('trend_slice_missing')
            details['trend_accuracy'] = None
        else:
            row = metrics.loc['trend_participation_target_state']
            details['trend_samples'] = int(row['samples'])
            details['trend_accuracy'] = float(row['accuracy'])
            if int(row['samples']) < TREND_SLICE_MIN_SAMPLES or float(row['accuracy']) < SPARSE_SLICE_ACCURACY_MIN:
                failed.append('trend_slice_accuracy')
    else:
        details['trend_samples'] = int(trend_samples)
        details['trend_accuracy'] = None
    details['failed_slice_checks'] = failed
    return len(failed) == 0, details


def metric_loss(row: pd.Series) -> float:
    rel = float(row.get('candidate_relative_return', float('nan')))
    dd = float(row.get('candidate_max_dd', float('nan')))
    rel_delta = float(row.get('relative_delta_vs_baseline', float('nan')))
    dd_delta = float(row.get('max_dd_delta_vs_baseline', float('nan')))
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / 30.0 if math.isfinite(rel) else 10.0
    loss += max(0.0, dd - TARGET_MAX_DD) / 15.0 if math.isfinite(dd) else 10.0
    loss += max(0.0, DRIFT_RELATIVE_FLOOR - rel_delta) / 10.0 if math.isfinite(rel_delta) else 5.0
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING) / 5.0 if math.isfinite(dd_delta) else 5.0
    if not bool(row.get('slice_gate', False)):
        loss += 1.0
    return float(loss)


In [4]:
init_qlib(QLIB_DATA_PATH)

fold_dataset_cache = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    fold_dataset_cache[fold_name] = {
        'dataset': dataset,
        'valid_df': segment_frame(dataset, 'valid'),
        'test_df': segment_frame(dataset, 'test'),
        'valid_price': price_slice(dataset, spec['valid']),
        'test_price': price_slice(dataset, spec['test']),
    }

fold_rows = []
slice_rows = []
action_rows = []
daily_rows = []

for item in CANDIDATES:
    candidate = item['candidate']
    set_seed(SEED + int(item['complexity']))
    targets = targets_by_source[item['target_source']]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold_targets = fold_target_subset(targets, spec)
        if fold_targets.empty:
            fold_rows.append({'candidate': candidate, 'fold': fold_name, 'status': 'skipped_no_targets'})
            continue
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=BC_TRAIN_CONFIG,
            sample_weight_config=item['sample_weight_config'],
            sample_size=0,
            rng_seed=SEED,
        )
        dataset_for_audit = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=SEED,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            dataset_for_audit.metadata,
            slice_cols=(
                'trend_participation_target_state',
                'is_rebound_cluster',
                'is_downtrend_state',
                'release_gate_state',
                'drawdown_control_state',
            ),
        )
        trend_samples = int(dataset_for_audit.metadata['trend_participation_target_state'].fillna(False).astype(bool).sum())
        slice_gate, slice_summary = fit_slice_gate(slice_metrics, trend_samples=trend_samples)
        tagged = slice_metrics.copy()
        tagged.insert(0, 'candidate', candidate)
        tagged.insert(1, 'fold', fold_name)
        slice_rows.extend(tagged.to_dict('records'))
        dist = fit.action_distribution.copy()
        dist.insert(0, 'candidate', candidate)
        dist.insert(1, 'fold', fold_name)
        action_rows.extend(dist.to_dict('records'))

        cache = fold_dataset_cache[fold_name]
        valid_metrics, valid_daily = evaluate_actor_path(
            fit.actor,
            cache['valid_df'],
            cache['valid_price'],
            profile,
            spec['valid'],
            label=f'{candidate}_{fold_name}_valid',
        )
        test_metrics, test_daily = evaluate_actor_path(
            fit.actor,
            cache['test_df'],
            cache['test_price'],
            profile,
            spec['test'],
            label=f'{candidate}_{fold_name}_test',
        )
        daily_rows.extend([
            {'candidate': candidate, 'fold': fold_name, 'segment': 'valid', 'path': str(valid_daily.relative_to(ROOT))},
            {'candidate': candidate, 'fold': fold_name, 'segment': 'test', 'path': str(test_daily.relative_to(ROOT))},
        ])
        fold_rows.append({
            'candidate': candidate,
            'fold': fold_name,
            'status': 'evaluated',
            'target_rows': int(len(fold_targets)),
            'bc_dataset_samples': int(len(dataset_for_audit.actions)),
            'trend_target_samples': int(trend_samples),
            'after_accuracy': float(fit.metrics['after_accuracy']),
            'after_loss': float(fit.metrics['after_loss']),
            'slice_gate': bool(slice_gate),
            'slice_failed_checks': json.dumps(slice_summary['failed_slice_checks'], sort_keys=True),
            'valid_relative_return': float(valid_metrics.get('relative_return', float('nan'))),
            'valid_max_dd': float(valid_metrics.get('max_dd', float('nan'))),
            'valid_avg_pos': float(valid_metrics.get('avg_pos', float('nan'))),
            'test_relative_return': float(test_metrics.get('relative_return', float('nan'))),
            'test_max_dd': float(test_metrics.get('max_dd', float('nan'))),
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_return': float(test_metrics.get('return', float('nan'))),
            'test_sharpe': float(test_metrics.get('sharpe', float('nan'))),
        })

fold_results = pd.DataFrame(fold_rows).sort_values(['candidate', 'fold']).reset_index(drop=True)
slice_metrics = pd.DataFrame(slice_rows).sort_values(['candidate', 'fold', 'slice']).reset_index(drop=True)
action_distribution = pd.DataFrame(action_rows).sort_values(['candidate', 'fold', 'action']).reset_index(drop=True)
daily_manifest = pd.DataFrame(daily_rows).sort_values(['candidate', 'fold', 'segment']).reset_index(drop=True)

fold_results.to_csv(CARD / 'bc_fold_results.csv', index=False)
slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
daily_manifest.to_csv(CARD / 'bc_daily_manifest.csv', index=False)

display(fold_results)


[13291:MainThread](2026-05-26 14:07:44,004) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13291:MainThread](2026-05-26 14:07:44,656) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13291:MainThread](2026-05-26 14:07:44,657) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[13291:MainThread](2026-05-26 14:07:46,955) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[13291:MainThread](2026-05-26 14:07:46,957) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,status,target_rows,bc_dataset_samples,trend_target_samples,after_accuracy,after_loss,slice_gate,slice_failed_checks,valid_relative_return,valid_max_dd,valid_avg_pos,test_relative_return,test_max_dd,test_avg_pos,test_return,test_sharpe
0,base_plain_contract,wf_2022,evaluated,253,253,0,0.833992,1.073453,True,[],-58.602980,12.185863,0.330955,36.419124,43.571257,0.280667,-40.195132,-1.551696
1,base_plain_contract,wf_2023_2024,evaluated,756,755,0,0.768212,1.146839,True,[],-38.468538,11.509663,0.272619,30.501116,18.630969,0.174542,-9.451594,-0.579104
2,trend_plain_contract,wf_2022,evaluated,253,253,0,0.833992,1.073453,True,[],-58.602980,12.185863,0.330955,36.419124,43.571257,0.280667,-40.195132,-1.551696
3,trend_plain_contract,wf_2023_2024,evaluated,756,755,15,0.754967,1.154651,True,[],-40.386846,10.785130,0.277116,41.557303,13.337480,0.189594,1.604592,0.206064
4,trend_weighted_obj025_cb099,wf_2022,evaluated,253,253,0,0.841897,1.069709,True,[],-61.691591,14.180729,0.323495,45.010860,35.085260,0.279286,-31.603396,-1.102362
5,trend_weighted_obj025_cb099,wf_2023_2024,evaluated,756,755,15,0.733775,1.189100,True,[],-54.265174,13.903899,0.260582,25.823105,21.353410,0.186649,-14.129605,-0.819902


In [5]:
evaluated = fold_results[fold_results['status'].eq('evaluated')].copy()
baseline_by_fold = {
    row['fold']: row
    for row in evaluated[evaluated['candidate'].eq(BASELINE_CANDIDATE)].to_dict('records')
}
score_rows = []
for row in evaluated.to_dict('records'):
    baseline = baseline_by_fold.get(row['fold'])
    if baseline is None:
        continue
    scored = score_bc_walkforward_drift(
        str(row['fold']),
        candidate_metrics={
            'relative_return': row['test_relative_return'],
            'max_dd': row['test_max_dd'],
            'avg_pos': row['test_avg_pos'],
        },
        baseline_metrics={
            'relative_return': baseline['test_relative_return'],
            'max_dd': baseline['test_max_dd'],
            'avg_pos': baseline['test_avg_pos'],
        },
        slice_gate=bool(row['slice_gate']),
        target_relative_return_pp=TARGET_RELATIVE_RETURN,
        target_max_dd_pct=TARGET_MAX_DD,
        relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR,
        max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING,
    )
    scored['candidate'] = row['candidate']
    scored['status'] = row['status']
    scored['valid_relative_return'] = row['valid_relative_return']
    scored['valid_max_dd'] = row['valid_max_dd']
    scored['valid_avg_pos'] = row['valid_avg_pos']
    scored['trend_target_samples'] = int(row['trend_target_samples'])
    scored['after_accuracy'] = float(row['after_accuracy'])
    scored['slice_failed_checks'] = row['slice_failed_checks']
    scored['failed_checks_json'] = json.dumps(scored['failed_checks'], sort_keys=True)
    score_rows.append(scored)

drift_scores = pd.DataFrame(score_rows)
if not drift_scores.empty:
    drift_scores['gate_loss'] = drift_scores.apply(metric_loss, axis=1)
    drift_scores = drift_scores.sort_values(['candidate', 'fold']).reset_index(drop=True)
drift_scores.to_csv(CARD / 'bc_drift_scores.csv', index=False)

summary_rows = []
required_folds = tuple(WALK_FORWARD_WINDOWS.keys())
for candidate, group in drift_scores.groupby('candidate', sort=True):
    summary = summarize_bc_walkforward_drift_screen(group, required_folds=required_folds)
    summary_rows.append({
        'candidate': candidate,
        'decision': summary['decision'],
        'fold_gate_pass_count': summary['fold_gate_pass_count'],
        'candidate_gate_pass_count': summary['candidate_gate_pass_count'],
        'baseline_gate_pass_count': summary['baseline_gate_pass_count'],
        'drift_gate_pass_count': summary['drift_gate_pass_count'],
        'slice_gate_pass_count': summary['slice_gate_pass_count'],
        'failed_folds_json': json.dumps(summary['failed_folds'], sort_keys=True),
        'gate_loss': float(group['gate_loss'].sum()) if 'gate_loss' in group else float('inf'),
        'min_relative_return': float(group['candidate_relative_return'].min()),
        'max_max_dd': float(group['candidate_max_dd'].max()),
        'mean_relative_delta_vs_baseline': float(group['relative_delta_vs_baseline'].mean()),
        'max_dd_delta_vs_baseline': float(group['max_dd_delta_vs_baseline'].max()),
        'promotion_allowed': False,
        'next_mechanism': summary['next_mechanism'],
    })

candidate_screen = pd.DataFrame(summary_rows).sort_values(
    ['fold_gate_pass_count', 'candidate_gate_pass_count', 'drift_gate_pass_count', 'gate_loss'],
    ascending=[False, False, False, True],
).reset_index(drop=True)
candidate_screen.to_csv(CARD / 'bc_candidate_screen.csv', index=False)

display(drift_scores)
display(candidate_screen)


,fold,candidate_relative_return,candidate_max_dd,candidate_avg_pos,baseline_relative_return,baseline_max_dd,baseline_avg_pos,relative_delta_vs_baseline,max_dd_delta_vs_baseline,candidate_gate,...,candidate,status,valid_relative_return,valid_max_dd,valid_avg_pos,trend_target_samples,after_accuracy,slice_failed_checks,failed_checks_json,gate_loss
0,wf_2022,36.419124,43.571257,0.280667,36.419124,43.571257,0.280667,0.000000,0.000000,False,...,base_plain_contract,evaluated,-58.602980,12.185863,0.330955,0,0.833992,[],"[""candidate_max_dd""]",1.904750
1,wf_2023_2024,30.501116,18.630969,0.174542,30.501116,18.630969,0.174542,0.000000,0.000000,False,...,base_plain_contract,evaluated,-38.468538,11.509663,0.272619,0,0.768212,[],"[""candidate_max_dd""]",0.242065
2,wf_2022,36.419124,43.571257,0.280667,36.419124,43.571257,0.280667,0.000000,0.000000,False,...,trend_plain_contract,evaluated,-58.602980,12.185863,0.330955,0,0.833992,[],"[""candidate_max_dd""]",1.904750
3,wf_2023_2024,41.557303,13.337480,0.189594,30.501116,18.630969,0.174542,11.056187,-5.293489,True,...,trend_plain_contract,evaluated,-40.386846,10.785130,0.277116,15,0.754967,[],[],0.000000
4,wf_2022,45.010860,35.085260,0.279286,36.419124,43.571257,0.280667,8.591736,-8.485997,False,...,trend_weighted_obj025_cb099,evaluated,-61.691591,14.180729,0.323495,0,0.841897,[],"[""candidate_max_dd""]",1.339017
5,wf_2023_2024,25.823105,21.353410,0.186649,30.501116,18.630969,0.174542,-4.678011,2.722441,False,...,trend_weighted_obj025_cb099,evaluated,-54.265174,13.903899,0.260582,15,0.733775,[],"[""candidate_relative_return"", ""candidate_max_d...",0.707279


,candidate,decision,fold_gate_pass_count,candidate_gate_pass_count,baseline_gate_pass_count,drift_gate_pass_count,slice_gate_pass_count,failed_folds_json,gate_loss,min_relative_return,max_max_dd,mean_relative_delta_vs_baseline,max_dd_delta_vs_baseline,promotion_allowed,next_mechanism
0,trend_plain_contract,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",1.904750,36.419124,43.571257,5.528093,0.000000,False,inspect_failed_fold_slices_or_checkpoint_selec...
1,base_plain_contract,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,2,"[""wf_2022"", ""wf_2023_2024""]",2.146815,30.501116,43.571257,0.000000,0.000000,False,revise_training_targets_before_ppo
2,trend_weighted_obj025_cb099,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,1,2,"[""wf_2022"", ""wf_2023_2024""]",2.046296,25.823105,35.085260,1.956863,2.722441,False,revise_training_targets_before_ppo


In [6]:
non_baseline = candidate_screen[~candidate_screen['candidate'].eq(BASELINE_CANDIDATE)].copy()
if non_baseline.empty:
    selected = candidate_screen.iloc[0].to_dict() if not candidate_screen.empty else {}
else:
    selected = non_baseline.iloc[0].to_dict()
selected_candidate = str(selected.get('candidate', ''))
selected_scores = drift_scores[drift_scores['candidate'].eq(selected_candidate)].copy() if selected_candidate else pd.DataFrame()
selected_summary = summarize_bc_walkforward_drift_screen(selected_scores, required_folds=tuple(WALK_FORWARD_WINDOWS.keys())) if not selected_scores.empty else {
    'decision': 'bc_walkforward_drift_no_rows',
    'promotion_allowed': False,
    'next_mechanism': 'repair_bc_screen_before_training',
}

all_selected_folds_pass = bool(selected_summary.get('decision') == 'bc_walkforward_drift_gate_passed_no_promotion')
any_selected_fold_pass = int(selected_summary.get('fold_gate_pass_count', 0)) > 0
any_candidate_gate = int(selected_summary.get('candidate_gate_pass_count', 0)) > 0
if all_selected_folds_pass:
    decision = 'trend_participation_bc_warmstart_passed_partial_wf_no_promotion'
    next_mechanism = 'run_checkpoint_or_short_ppo_screen_with_drawdown_drift_stop_then_fixed_split_wf2020'
elif any_selected_fold_pass:
    decision = 'trend_participation_bc_warmstart_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_trend_slice_and_checkpoint_selection_before_ppo'
elif any_candidate_gate:
    decision = 'trend_participation_bc_warmstart_drawdown_drift_failed_no_promotion'
    next_mechanism = 'block_ppo_until_trend_target_drawdown_drift_is_fixed'
else:
    decision = 'trend_participation_bc_warmstart_failed_no_promotion'
    next_mechanism = 'revise_trend_target_or_add_drawdown_state_before_ppo'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_candidate': selected_candidate,
    'selected_summary': selected,
    'selected_walkforward_summary': selected_summary,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'trend_participation_foldlocal_target_v1',
        'base_plain_contract',
    ],
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'training_target', 'bc_warm_start'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING,
    'folds_evaluated': list(WALK_FORWARD_WINDOWS.keys()),
    'wf_2020_coverage': False,
    'wf_2020_skip_reason': 'target rows start in 2020; using them would leak wf_2020 labels',
    'promotion_allowed': False,
    'promotion_blocker': 'This is a diagnostic BC warm-start screen. Promotion requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 replay and an actor retention decision.',
    'ppo_screen_allowed_next': bool(all_selected_folds_pass),
    'training_allowed_from_this_card': bool(all_selected_folds_pass),
    'actor_retention': 'no actor bundle retained; rerun notebook to regenerate diagnostic BC actors',
    'next_mechanism': next_mechanism,
    'source_selection': source_selection,
    'external_sources': EXTERNAL_SOURCES,
    'sidecars': [
        'bc_candidate_configs.csv',
        'coverage.csv',
        'coverage.json',
        'bc_fold_results.csv',
        'bc_slice_metrics.csv',
        'bc_action_distribution.csv',
        'bc_daily_manifest.csv',
        'bc_drift_scores.csv',
        'bc_candidate_screen.csv',
        '*_daily_path.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selected_candidate': selected_candidate,
    'promotion_allowed': False,
    'ppo_screen_allowed_next': bool(all_selected_folds_pass),
    'candidate_screen': candidate_screen.to_dict('records'),
    'selected_scores': selected_scores.to_dict('records'),
    'conclusion': [],
    'next_mechanism': next_mechanism,
}
if selected_scores.empty:
    summary['conclusion'].append('No selected walk-forward rows were available; do not train further from this card.')
else:
    for row in selected_scores.sort_values('fold').to_dict('records'):
        summary['conclusion'].append(
            f"{row['fold']}: {selected_candidate} relative={row['candidate_relative_return']:.2f}pp, "
            f"MaxDD={row['candidate_max_dd']:.2f}%, delta vs base rel={row['relative_delta_vs_baseline']:.2f}pp, "
            f"delta MaxDD={row['max_dd_delta_vs_baseline']:.2f}pp, fold_gate={bool(row['fold_gate'])}."
        )
    summary['conclusion'].append(
        'No runtime promotion: this card evaluates BC warm-start behavior only and skips wf_2020 by leakage guard.'
    )
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "trend_participation_bc_warmstart_screen_v1",
  "created_at": "2026-05-26T06:09:24.365310+00:00",
  "decision": "trend_participation_bc_warmstart_mixed_no_promotion",
  "selected_candidate": "trend_plain_contract",
  "selected_summary": {
    "candidate": "trend_plain_contract",
    "decision": "bc_walkforward_drift_mixed_no_promotion",
    "fold_gate_pass_count": 1,
    "candidate_gate_pass_count": 1,
    "baseline_gate_pass_count": 0,
    "drift_gate_pass_count": 2,
    "slice_gate_pass_count": 2,
    "failed_folds_json": "[\"wf_2022\"]",
    "gate_loss": 1.9047504488370985,
    "min_relative_return": 36.41912423582883,
    "max_max_dd": 43.57125673255648,
    "mean_relative_delta_vs_baseline": 5.528093458113453,
    "max_dd_delta_vs_baseline": 0.0,
    "promotion_allowed": false,
    "next_mechanism": "inspect_failed_fold_slices_or_checkpoint_selection_before_ppo"
  },
  "selected_walkforward_summary": {
    "decision": "bc_walkforward_drift_mixed_no_promotion",
  

In [7]:
selection = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
summary = json.loads((CARD / 'summary.json').read_text(encoding='utf-8'))
conclusion_lines = '\n'.join(f"- {line}" for line in summary['conclusion'])
selected = selection.get('selected_summary', {})
summary_md = f"""
## Runtime Conclusion

Decision: `{selection['decision']}`. Promotion is blocked.

Selected candidate: `{selection['selected_candidate']}`.

Selected aggregate: fold gates `{selected.get('fold_gate_pass_count')}`, candidate gates `{selected.get('candidate_gate_pass_count')}`, drift gates `{selected.get('drift_gate_pass_count')}`, min relative `{selected.get('min_relative_return')}`, max MaxDD `{selected.get('max_max_dd')}`.

{conclusion_lines}

Next action: `{selection['next_mechanism']}`.
""".strip()
display(Markdown(summary_md))
candidate_screen


## Runtime Conclusion

Decision: `trend_participation_bc_warmstart_mixed_no_promotion`. Promotion is blocked.

Selected candidate: `trend_plain_contract`.

Selected aggregate: fold gates `1`, candidate gates `1`, drift gates `2`, min relative `36.41912423582883`, max MaxDD `43.57125673255648`.

- wf_2022: trend_plain_contract relative=36.42pp, MaxDD=43.57%, delta vs base rel=0.00pp, delta MaxDD=0.00pp, fold_gate=False.
- wf_2023_2024: trend_plain_contract relative=41.56pp, MaxDD=13.34%, delta vs base rel=11.06pp, delta MaxDD=-5.29pp, fold_gate=True.
- No runtime promotion: this card evaluates BC warm-start behavior only and skips wf_2020 by leakage guard.

Next action: `inspect_failed_fold_trend_slice_and_checkpoint_selection_before_ppo`.

,candidate,decision,fold_gate_pass_count,candidate_gate_pass_count,baseline_gate_pass_count,drift_gate_pass_count,slice_gate_pass_count,failed_folds_json,gate_loss,min_relative_return,max_max_dd,mean_relative_delta_vs_baseline,max_dd_delta_vs_baseline,promotion_allowed,next_mechanism
0,trend_plain_contract,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",1.904750,36.419124,43.571257,5.528093,0.000000,False,inspect_failed_fold_slices_or_checkpoint_selec...
1,base_plain_contract,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,2,"[""wf_2022"", ""wf_2023_2024""]",2.146815,30.501116,43.571257,0.000000,0.000000,False,revise_training_targets_before_ppo
2,trend_weighted_obj025_cb099,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,1,2,"[""wf_2022"", ""wf_2023_2024""]",2.046296,25.823105,35.085260,1.956863,2.722441,False,revise_training_targets_before_ppo
